# Teste de implementação utilizando regressão

## Libs

In [ ]:
import pandas as pd
import numpy as np

import plotly.express as px
import plotly.graph_objects as go

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

import warnings
warnings.filterwarnings('ignore')

## Load and preprocess Dataset

In [ ]:
df_dataset = pd.read_csv('data/Depurador 762-28-006 - Cozimento.csv', sep=';', decimal='.', encoding='utf-8-sig')
df_dataset.drop(columns=['762-34-073.CR'], inplace=True)
df_dataset['Timestamp'] = pd.to_datetime(df_dataset['Timestamp'], format='%Y-%m-%d %H:%M:%S')

for col in df_dataset.select_dtypes(include=['object']).columns:
    df_dataset[col] = pd.to_numeric(df_dataset[col], errors='coerce')

df_dataset.dropna(inplace=True)
print("Dataset shape after dropping missing values:", df_dataset.shape)
df_dataset.head()

## Train/Test Split

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=df_dataset['Timestamp'], y=df_dataset['762P0034.PV'], mode='lines', name='762P0034.PV'))
fig.update_layout(title='762P0034.PV over Time', xaxis_title='Timestamp', yaxis_title='762P0034.PV')
fig.show()

In [ ]:
target = '762P0034.PV'
features = [df for df in df_dataset.columns if df != target and df != 'Timestamp']

## Train 1
start_date_train = pd.to_datetime('2024-01-01 00:00:00')
end_date_train = pd.to_datetime('2024-04-15 00:00:00')
mask = (df_dataset['Timestamp'] >= start_date_train) & (df_dataset['Timestamp'] <= end_date_train)
df_train1 = df_dataset.loc[mask]

## Train 2
start_date_train2 = pd.to_datetime('2024-12-15 00:00:00')
end_date_train2 = pd.to_datetime('2024-02-18 00:00:00')
mask = (df_dataset['Timestamp'] >= start_date_train2) & (df_dataset['Timestamp'] <= end_date_train2)
df_train2 = df_dataset.loc[mask]

## Train 3
start_date_train3 = pd.to_datetime('2025-06-18 00:00:00')
end_date_train3 = pd.to_datetime('2025-09-19 00:00:00')
mask = (df_dataset['Timestamp'] >= start_date_train3) & (df_dataset['Timestamp'] <= end_date_train3)
df_train3 = df_dataset.loc[mask]

df_train = pd.concat([df_train1, df_train2, df_train3])
X_train = df_train[features]
y_train = df_train[target]

X_all = df_dataset[features] # Todo o período para teste
y_all = df_dataset[target]

## Regressor

In [ ]:
# Regressor para estimar a variável crítica baseada nas demais
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Predição em toda a série (passado e futuro/degradação)
df_dataset['762P0034.PV_predict'] = model.predict(X_all)
df_train['762P0034.PV_predict'] = model.predict(X_train)

## Residual Analisys

In [ ]:
# O resíduo é a diferença entre o Comportamento Real e o Estimado pelo modelo
df_dataset['residual'] = df_dataset['762P0034.PV'] - df_dataset['762P0034.PV_predict']

# Suavização do resíduo (Média Móvel) para evitar falsos alarmes por ruído
df_dataset['residuo_smooth'] = df_dataset['residual'].rolling(window=24).mean()

# Definição de um Limiar de Alerta (Baseado no desvio padrão do treino)
df_train['residual'] = df_train['762P0034.PV'] - df_train['762P0034.PV_predict']
threshold = df_train['residual'].std() * 3  # 3 desvios padrão

# Identificar pontos onde o modelo não explica mais a variável (Degradação)
df_dataset['alerta_degradacao'] = df_dataset['residuo_smooth'] > threshold

## Visualização dos Resultados
fig = go.Figure()
fig.add_trace(go.Scatter(x=df_dataset['Timestamp'], y=df_dataset['762P0034.PV'], mode='lines', name='762P0034.PV Real'))
fig.add_trace(go.Scatter(x=df_dataset['Timestamp'], y=df_dataset['762P0034.PV_predict'], mode='lines', name='762P0034.PV Predito'))
fig.add_trace(go.Scatter(x=df_dataset['Timestamp'], y=df_dataset['residuo_smooth'], mode='lines', name='Resíduo Suavizado', yaxis='y2'))
fig.add_trace(go.Scatter(x=df_dataset['Timestamp'], y=[threshold]*len(df_dataset), mode='lines', name='Limiar de Alerta', yaxis='y2', line=dict(dash='dash')))
fig.update_layout(
    title='Monitoramento de 762P0034.PV com Detecção de Degradação',
    xaxis_title='Timestamp',
    yaxis_title='762P0034.PV',
    yaxis2=dict(
        title='Resíduo Suavizado',
        overlaying='y',
        side='right'
    )
)
fig.show()
# fig.write_html('output/monitoramento_762P0034_PV.html')

## Forecasting

In [ ]:
# steps to forecast ahead
steps_ahead = 60 

# target shifted
# model: Features(t) -> Target(t + steps_ahead)
df_dataset['target_future'] = df_dataset[target].shift(-steps_ahead)

df_model = df_dataset.dropna(subset=['target_future']).copy()

# Atualiza as features (garantindo que o target futuro não vaze para as features)
features = [col for col in df_model.columns if col not in [target, 'target_future', 'Timestamp', '762-34-073.CR']]

# train test split based on date ranges
mask_train = (
    ((df_model['Timestamp'] >= start_date_train) & (df_model['Timestamp'] <= end_date_train)) |
    ((df_model['Timestamp'] >= start_date_train2) & (df_model['Timestamp'] <= end_date_train2)) |
    ((df_model['Timestamp'] >= start_date_train3) & (df_model['Timestamp'] <= end_date_train3))
)
X_train = df_model.loc[mask_train, features]
y_train = df_model.loc[mask_train, 'target_future'] # Treina mirando no futuro

X_all = df_model[features]
y_all_future_real = df_model['target_future']

# model
model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

# Esta predição agora significa: "Com as condições de AGORA, qual será a pressão daqui a 60 passos?"
df_model['prediction_future'] = model.predict(X_all)

# O resíduo agora indica erro de prognóstico
df_model['residual_forecast'] = df_model['target_future'] - df_model['prediction_future']

# Visualização rápida
import plotly.graph_objects as go
fig = go.Figure()
fig.add_trace(go.Scatter(x=df_model['Timestamp'], y=df_model['target_future'], name=f'Real em t+{steps_ahead}'))
fig.add_trace(go.Scatter(x=df_model['Timestamp'], y=df_model['prediction_future'], name=f'Previsão para t+{steps_ahead}'))
fig.show()